In [2]:
#Only if you want to download the csv files
import kagglehub
import shutil
import os

# Download dataset to Kaggle cache
path = kagglehub.dataset_download("mishra5001/credit-card")

# Copy dataset into current folder
destination = os.path.join(os.getcwd(), "credit_card_data")

shutil.copytree(path, destination, dirs_exist_ok=True)

print("Original download path:", path)
print("Copied dataset to:", destination)

Original download path: C:\Users\User\.cache\kagglehub\datasets\mishra5001\credit-card\versions\1
Copied dataset to: c:\Users\User\Documents\2026 SCHOOL_MSc\Adaptive CML\PROJECT\ACML_26_P1\Datasets\credit_card_data


The dataset is converted from CSV to Parquet format to improve storage efficiency and reduce memory consumption during preprocessing. The Parquet format also improves data loading performance and preserved feature data types more effectively than CSV files.

In [3]:
import pandas as pd

# Load original CSV
df = pd.read_csv("credit_card_data/application_data.csv")

# Save as parquet
df.to_parquet("application_data.parquet", engine="pyarrow", index=False)

print("Parquet file created successfully")

Parquet file created successfully


In [ ]:
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer


#load the parquet file:
df = pd.read_parquet('application_data.parquet', engine='pyarrow')

In [5]:
#Check structure
print(df.head())
print(df.info())
print(df.shape)

   SK_ID_CURR  TARGET NAME_CONTRACT_TYPE CODE_GENDER FLAG_OWN_CAR  \
0      100002       1         Cash loans           M            N   
1      100003       0         Cash loans           F            N   
2      100004       0    Revolving loans           M            Y   
3      100006       0         Cash loans           F            N   
4      100007       0         Cash loans           M            N   

  FLAG_OWN_REALTY  CNT_CHILDREN  AMT_INCOME_TOTAL  AMT_CREDIT  AMT_ANNUITY  \
0               Y             0          202500.0    406597.5      24700.5   
1               N             0          270000.0   1293502.5      35698.5   
2               Y             0           67500.0    135000.0       6750.0   
3               Y             0          135000.0    312682.5      29686.5   
4               Y             0          121500.0    513000.0      21865.5   

   ...  FLAG_DOCUMENT_18 FLAG_DOCUMENT_19 FLAG_DOCUMENT_20 FLAG_DOCUMENT_21  \
0  ...                 0             

In [6]:
#Check missing values
print(df.isnull().sum())

SK_ID_CURR                        0
TARGET                            0
NAME_CONTRACT_TYPE                0
CODE_GENDER                       0
FLAG_OWN_CAR                      0
                              ...  
AMT_REQ_CREDIT_BUREAU_DAY     41519
AMT_REQ_CREDIT_BUREAU_WEEK    41519
AMT_REQ_CREDIT_BUREAU_MON     41519
AMT_REQ_CREDIT_BUREAU_QRT     41519
AMT_REQ_CREDIT_BUREAU_YEAR    41519
Length: 122, dtype: int64


In [7]:
# Removing duplicates
print(df.shape)
df = df.drop_duplicates()
print(df.shape)

(307511, 122)
(307511, 122)


Feature selection is performed prior to modelling to eliminate irrelevant and low-quality variables. Columns with excessive missing values, unique identifier attributes, and features with negligible variance are removed to improve model efficiency and reduce noise. Missing value analysis conducted to determine suitable preprocessing strategies for the remaining attributes.

In [8]:
#Check Missing Percentage Per Column
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': missing_percent.index,
    'Missing %': missing_percent.values
})

missing_df = missing_df.sort_values(by='Missing %', ascending=False)

print(missing_df.head(20))

                      Column  Missing %
76           COMMONAREA_MEDI  69.872297
48            COMMONAREA_AVG  69.872297
62           COMMONAREA_MODE  69.872297
70  NONLIVINGAPARTMENTS_MODE  69.432963
56   NONLIVINGAPARTMENTS_AVG  69.432963
84  NONLIVINGAPARTMENTS_MEDI  69.432963
86        FONDKAPREMONT_MODE  68.386172
68     LIVINGAPARTMENTS_MODE  68.354953
54      LIVINGAPARTMENTS_AVG  68.354953
82     LIVINGAPARTMENTS_MEDI  68.354953
52             FLOORSMIN_AVG  67.848630
66            FLOORSMIN_MODE  67.848630
80            FLOORSMIN_MEDI  67.848630
75          YEARS_BUILD_MEDI  66.497784
61          YEARS_BUILD_MODE  66.497784
47           YEARS_BUILD_AVG  66.497784
21               OWN_CAR_AGE  65.990810
81             LANDAREA_MEDI  59.376738
67             LANDAREA_MODE  59.376738
53              LANDAREA_AVG  59.376738


In [ ]:
# Drop columns with >50% missing values
cols_to_drop = missing_percent[missing_percent > 50].index

print(cols_to_drop)

df = df.drop(columns=cols_to_drop)

print(df.shape)

Index(['OWN_CAR_AGE', 'EXT_SOURCE_1', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG',
       'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG',
       'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG',
       'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG',
       'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'YEARS_BUILD_MODE',
       'COMMONAREA_MODE', 'ELEVATORS_MODE', 'ENTRANCES_MODE', 'FLOORSMIN_MODE',
       'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE',
       'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE', 'APARTMENTS_MEDI',
       'BASEMENTAREA_MEDI', 'YEARS_BUILD_MEDI', 'COMMONAREA_MEDI',
       'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI',
       'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI', 'NONLIVINGAPARTMENTS_MEDI',
       'NONLIVINGAREA_MEDI', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE',
       'WALLSMATERIAL_MODE'],
      dtype='object')
(307511, 81)


In [ ]:
# Remove ID Columns
df = df.drop(columns=['SK_ID_CURR'])

print(df.shape)

In [16]:
# Check Columns With Only One Value

low_variance = []

for col in df.columns:
    if df[col].nunique() <= 1:
        low_variance.append(col)

print(low_variance)

df = df.drop(columns=low_variance)

print(df.shape)

[]
(307511, 80)


In [17]:
summary = pd.DataFrame({
    'Data Type': df.dtypes,
    'Missing Values': df.isnull().sum(),
    'Unique Values': df.nunique()
})

print(summary)

                           Data Type  Missing Values  Unique Values
TARGET                         int64               0              2
NAME_CONTRACT_TYPE            object               0              2
CODE_GENDER                   object               0              3
FLAG_OWN_CAR                  object               0              2
FLAG_OWN_REALTY               object               0              2
...                              ...             ...            ...
AMT_REQ_CREDIT_BUREAU_DAY    float64           41519              9
AMT_REQ_CREDIT_BUREAU_WEEK   float64           41519              9
AMT_REQ_CREDIT_BUREAU_MON    float64           41519             24
AMT_REQ_CREDIT_BUREAU_QRT    float64           41519             11
AMT_REQ_CREDIT_BUREAU_YEAR   float64           41519             25

[80 rows x 3 columns]


In [19]:
# Separate numerical and categorical columns
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include=["object"]).columns


In [ ]:
#Imputation - skip
# Fill numerical missing values column by column using median
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical missing values column by column using mode
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values remaining:")
print(df.isnull().sum().sum())

In [20]:
#Encode Categorical Variables (Convert categories into numbers)
encoder = LabelEncoder()

for col in cat_cols: 
    df[col] = encoder.fit_transform(df[col])

In [22]:
# Feature Scaling

X = df.drop("TARGET", axis=1)
y = df["TARGET"]

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [23]:
# Train/Test Split

X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled,
    y,
    test_size=0.30,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)